# TextMamba3D — A100 Training Pipeline (v4.6)

**V4.6: AttnRes-inspired Cross-Scale Skip Attention + Text Scale Gate**

| Feature | Description |
|---------|-------------|
| Direction A | CrossScaleSkipAttention supplements decoder skip connections with cross-scale context |
| Direction B | TextScaleGate adaptively mixes raw vs text-fused features per scale |
| A100 40GB | batch_size=4, gradient_checkpointing=true, sw_batch_size=2, num_workers=4 |

Config: `configs/textbrats_v8.yaml`

In [15]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# GPU check + install dependencies
!nvidia-smi 2>/dev/null || echo "No GPU detected (CPU mode)"
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache mamba-ssm causal-conv1d transformers nibabel tensorboard pyyaml tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thu Mar 19 00:05:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|          

In [ ]:
import os, zipfile, shutil, subprocess, time

REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'

# Code source: git clone (public repo)
# If old zip-extracted dir exists (no .git), remove and re-clone
git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    print(f'Removing old non-git code at {REPO_DIR}...')
    shutil.rmtree(REPO_DIR)

if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
    print(f'Updated existing repo at {REPO_DIR}')
else:
    # Clone with retry
    for attempt in range(1, 4):
        print(f'Cloning (attempt {attempt}/3)...')
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True
        )
        if ret.returncode == 0 and os.path.exists(os.path.join(REPO_DIR, 'models/textmamba3d.py')):
            break
        print(f'  Failed (code {ret.returncode}): {ret.stderr.strip()}')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        if attempt < 3:
            time.sleep(5 * attempt)
    else:
        raise RuntimeError(f'Clone failed after 3 attempts. Last error: {ret.stderr.strip()}')
    os.chdir(REPO_DIR)
    print(f'Cloned to {REPO_DIR}')

print(f'Working directory: {os.getcwd()}')

# Extract BraTS data from Drive
DATA_ZIP = os.path.join(DRIVE_BASE, "TextBraTS_data.zip")
DATA_DIR = os.path.join(REPO_DIR, "data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData")

if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    if os.path.exists(DATA_ZIP):
        print(f"Extracting {DATA_ZIP}...")
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(os.path.dirname(DATA_DIR))
        if os.path.exists(DATA_DIR):
            print(f"Data extracted. Cases: {len(os.listdir(DATA_DIR))}")
        else:
            print(f"ERROR: Expected path not found after extraction: {DATA_DIR}")
            print("Actual contents:", os.listdir(os.path.dirname(DATA_DIR)))
    else:
        print(f"ERROR: {DATA_ZIP} not found on Drive")
else:
    print(f"Data already exists. Cases: {len(os.listdir(DATA_DIR))}")

# Count samples
if os.path.exists(DATA_DIR):
    cases = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"Total BraTS cases: {len(cases)}")

In [ ]:
import sys, yaml, inspect
sys.path.insert(0, '.')

# Verify V4.6 modules
from models.fusion import (
    SequentialCrossAttention, MultiScaleSeqCA,
    RMSNorm, CrossScaleSkipAttention, TextScaleGate, MultiScaleTextGate,
)
print("V4.4 modules: SequentialCrossAttention, MultiScaleSeqCA")
print("V4.6 modules: RMSNorm, CrossScaleSkipAttention, TextScaleGate, MultiScaleTextGate")

# Verify decoder has CrossScaleSkipAttention support
from models.decoder_3d import MambaDecoder3D
sig = inspect.signature(MambaDecoder3D.__init__)
assert 'use_cross_scale_skip' in sig.parameters, "decoder missing use_cross_scale_skip param!"
print("Decoder: use_cross_scale_skip parameter present")

# Verify textmamba3d has V4.6 params
from models.textmamba3d import TextMamba3D
sig = inspect.signature(TextMamba3D.__init__)
for param in ['use_text_gate', 'use_cross_scale_skip', 'text_gate_init_bias']:
    assert param in sig.parameters, f"TextMamba3D missing {param}!"
print("TextMamba3D: use_text_gate, use_cross_scale_skip, text_gate_init_bias present")

# Verify config
with open('configs/textbrats_v8.yaml') as f:
    cfg = yaml.safe_load(f)
assert cfg['model']['use_cross_scale_skip'] is True
assert cfg['model']['use_text_gate'] is True
assert cfg['data']['batch_size'] == 4, "A100 batch_size should be 4"
assert cfg['training']['gradient_checkpointing'] is True, "A100 should use grad ckpt"
print("Config: V4.6 features enabled, A100 optimizations confirmed")

print()
print("All V4.6 modules verified!")

In [ ]:
import os, sys, zipfile
os.chdir(REPO_DIR)

DATA_DIR = "./data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
ET_CACHE_ZIP = os.path.join(DRIVE_BASE, "et_enriched.zip")

# Check if already in data dir
sample_case = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))[0]
sample_enriched = os.path.join(DATA_DIR, sample_case, f"{sample_case}_et_enriched.txt")

if os.path.exists(sample_enriched):
    # Already generated (same runtime)
    count = sum(
        1 for d in os.listdir(DATA_DIR)
        if os.path.isdir(os.path.join(DATA_DIR, d))
        and os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt"))
    )
    print(f"ET-enriched text already present for {count} cases, skipping")
    for case in sorted(os.listdir(DATA_DIR))[:3]:
        path = os.path.join(DATA_DIR, case, f"{case}_et_enriched.txt")
        if os.path.exists(path):
            with open(path, 'r') as f:
                print(f"  {case}: {f.read().strip()[:120]}...")

elif os.path.exists(ET_CACHE_ZIP):
    # Restore from Drive cache
    print(f"Restoring ET-enriched text from {ET_CACHE_ZIP}...")
    with zipfile.ZipFile(ET_CACHE_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    count = sum(
        1 for d in os.listdir(DATA_DIR)
        if os.path.isdir(os.path.join(DATA_DIR, d))
        and os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt"))
    )
    print(f"Restored ET-enriched text for {count} cases from Drive cache")

else:
    # Generate + cache to Drive
    print("Generating ET-enriched text descriptions from T1ce images...")
    sys.path.insert(0, '.')
    from data.et_text_enrichment import process_all_cases
    results = process_all_cases(DATA_DIR)

    no_enhancement = sum(1 for desc in results.values() if "No significant" in desc)
    total = len(results)
    print(f"Total: {total}, No enhancement: {no_enhancement} ({no_enhancement/total*100:.1f}%)")

    # Cache to Drive for next runtime
    with zipfile.ZipFile(ET_CACHE_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
        for case_dir in sorted(os.listdir(DATA_DIR)):
            case_path = os.path.join(DATA_DIR, case_dir)
            if not os.path.isdir(case_path):
                continue
            et_file = os.path.join(case_path, f"{case_dir}_et_enriched.txt")
            if os.path.exists(et_file):
                zf.write(et_file, os.path.join(case_dir, f"{case_dir}_et_enriched.txt"))
    print(f"Cached ET text to {ET_CACHE_ZIP}")

## Training (A100 40GB)

| Parameter | Value |
|-----------|-------|
| batch_size | 4 |
| gradient_checkpointing | true |
| sw_batch_size | 2 |
| num_workers | 4 |
| gradient_accumulation | 1 |

In [ ]:
import os, shutil, glob

DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
os.makedirs(DRIVE_CKPT, exist_ok=True)

def sync_checkpoints_to_drive():
    local_ckpt = os.path.join(REPO_DIR, "checkpoints")
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, "*.pth")):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(f))
        shutil.copy2(f, dst)
    print(f"Synced checkpoints to {DRIVE_CKPT}")

# Clean local checkpoints (fresh start for v4.6)
for f in glob.glob(os.path.join(REPO_DIR, "checkpoints/*.pth")):
    os.remove(f)
print("Cleaned local checkpoints for v4.6 fresh start")
print("(Previous best checkpoints preserved on Drive)")


In [ ]:
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

# V4.6 training: SeqCA + CrossScaleSkipAttention + TextScaleGate + ET-Enriched (A100 40GB)
!python -u train.py \
    --config configs/textbrats_v8.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 1 \
    2>&1 | tee training_v4.6_a100.log | grep --line-buffered -E "(Epoch [0-9]+:|train_loss=|Best |Error|Traceback)"

# Sync and save
sync_checkpoints_to_drive()

best_ckpt = os.path.join(DRIVE_CKPT, "best_v4.6.pth")
local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
if os.path.exists(local_best):
    shutil.copy2(local_best, best_ckpt)
    print(f"Best checkpoint saved: {best_ckpt}")

## Evaluation


In [ ]:
os.chdir(REPO_DIR)

ckpt = os.path.join(REPO_DIR, "checkpoints/best.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, "best_v4.6.pth")

if os.path.exists(ckpt):
    print("=" * 60)
    print("Evaluation: With Text (SeqCA + TextScaleGate + CrossScaleSkip)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_v8.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --use-text \
        --overlap 0.5

    print()

    print("=" * 60)
    print("Evaluation: Without Text (fusion bypassed)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_v8.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --no-text \
        --overlap 0.5

    print()
    print("=" * 60)
    print("Compare: with-text Dice - without-text Dice = text guidance delta")
    print("V4.5 baseline: Mean Dice 83.48%")
    print("V4.6 target: >= 84%")
    print("=" * 60)
else:
    print(f"No checkpoint found at {ckpt}")
    print("Run training first")

In [ ]:
import matplotlib.pyplot as plt

# Placeholder: fill in actual results after training
v45_dice = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 83.48}
v46_dice = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 0.0}  # Fill after eval

if v46_dice['Mean'] == 0.0:
    print("V4.6 results not yet filled in.")
    print("Update v45_dice and v46_dice dictionaries after evaluation, then re-run this cell.")
else:
    labels = list(v45_dice.keys())
    v45_vals = list(v45_dice.values())
    v46_vals = list(v46_dice.values())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart comparison
    x = range(len(labels))
    w = 0.35
    ax1.bar([i - w/2 for i in x], v45_vals, w, label='V4.5', color='steelblue', alpha=0.8)
    ax1.bar([i + w/2 for i in x], v46_vals, w, label='V4.6 (A100)', color='coral', alpha=0.8)
    ax1.set_ylabel('Dice (%)')
    ax1.set_title('V4.5 vs V4.6 Dice Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    # Delta chart
    deltas = [v46 - v45 for v45, v46 in zip(v45_vals, v46_vals)]
    colors = ['green' if d >= 0 else 'red' for d in deltas]
    ax2.bar(labels, deltas, color=colors, alpha=0.8)
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_ylabel('Delta (%)')
    ax2.set_title('V4.6 - V4.5 Improvement')
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('v46_a100_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: v46_a100_comparison.png")

## Resume Training (After Disconnect)


In [ ]:
import os, shutil
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

resume_ckpt = os.path.join(DRIVE_CKPT, "last.pth")
if os.path.exists(resume_ckpt):
    print(f"Resuming from {resume_ckpt}")
    !python train.py \
        --config configs/textbrats_v8.yaml \
        --resume "{resume_ckpt}" \
        --no-text-ratio 0.15 \
        --grad-accum 1

    sync_checkpoints_to_drive()

    best_ckpt = os.path.join(DRIVE_CKPT, "best_v4.6.pth")
    local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
    if os.path.exists(local_best):
        shutil.copy2(local_best, best_ckpt)
        print(f"Best checkpoint saved: {best_ckpt}")
else:
    print("No checkpoint to resume from.")
    print(f"Expected: {resume_ckpt}")
    print("Run training first")